# S&P 500 Stock Market Analysis
## Exploratory Data Analysis (EDA) - Portfolio Project

---

**Analysis Period:** March 2025 - February 2026  
**Stocks Analyzed:** 50 Top S&P 500 Companies  
**Data Source:** Yahoo Finance API

---


## 1. Import Libraries

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

## 2. Data Acquisition

In [ ]:
# Define analysis parameters
END_DATE = datetime.now()
START_DATE = END_DATE - timedelta(days=365)

# Top S&P 500 companies
TICKERS = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'BRK-B', 'UNH', 'JNJ',
    'V', 'XOM', 'JPM', 'PG', 'MA', 'HD', 'CVX', 'LLY', 'ABBV', 'MRK',
    'PEP', 'KO', 'COST', 'AVGO', 'TMO', 'WMT', 'MCD', 'CSCO', 'ACN', 'DIS',
    'ABT', 'DHR', 'CRM', 'WFC', 'NEE', 'TXN', 'NKE', 'PM', 'UNP', 'BA',
    'HON', 'LOW', 'INTC', 'UPS', 'IBM', 'CAT', 'SBUX', 'AMD', 'GE', 'INTU'
]

print(f"Analysis Period: {START_DATE.strftime('%Y-%m-%d')} to {END_DATE.strftime('%Y-%m-%d')}")
print(f"Number of Stocks: {len(TICKERS)}")

In [ ]:
# Download stock data
stock_data = yf.download(TICKERS, start=START_DATE, end=END_DATE, progress=True)

# Extract adjusted close prices
adj_close = stock_data['Close']  # Using Close as Adj Close equivalent
volume = stock_data['Volume']

print(f"\nData Shape: {adj_close.shape}")
print(f"Date Range: {adj_close.index.min()} to {adj_close.index.max()}")

## 3. Data Cleaning & Preprocessing

In [ ]:
# Fill missing values
adj_close_filled = adj_close.fillna(method='ffill').fillna(method='bfill')
volume_filled = volume.fillna(0)

print("Data Quality Summary:")
print(f"- Trading Days: {len(adj_close_filled)}")
print(f"- Number of Stocks: {len(adj_close_filled.columns)}")
print(f"- Missing Values: {adj_close_filled.isnull().sum().sum()}")

## 4. Performance Metrics Calculation

In [ ]:
# Calculate daily returns
daily_returns = adj_close_filled.pct_change().dropna()

# Create metrics DataFrame
metrics = pd.DataFrame(index=adj_close_filled.columns)

# Total Return
metrics['Total Return (%)'] = ((adj_close_filled.iloc[-1] / adj_close_filled.iloc[0]) - 1) * 100

# Annualized Return
days = len(adj_close_filled)
metrics['Annualized Return (%)'] = ((1 + metrics['Total Return (%)']/100) ** (252/days) - 1) * 100

# Volatility (Annualized)
metrics['Volatility (%)'] = daily_returns.std() * np.sqrt(252) * 100

# Sharpe Ratio (risk-free rate = 4%)
risk_free_rate = 0.04
metrics['Sharpe Ratio'] = (metrics['Annualized Return (%)']/100 - risk_free_rate) / (metrics['Volatility (%)']/100)

# Max Drawdown
def max_drawdown(prices):
    peak = prices.cummax()
    drawdown = (prices - peak) / peak
    return drawdown.min() * 100

metrics['Max Drawdown (%)'] = adj_close_filled.apply(max_drawdown)

# Average Volume
metrics['Avg Volume (M)'] = volume_filled.mean() / 1e6

# Sort by return
metrics_sorted = metrics.sort_values('Total Return (%)', ascending=False)

print("Top 10 Performers:")
metrics_sorted[['Total Return (%)', 'Annualized Return (%)', 'Volatility (%)', 'Sharpe Ratio']].head(10)

## 5. Visualizations

In [ ]:
# Create output directory
import os
output_dir = 'sp500_analysis_plots'
os.makedirs(output_dir, exist_ok=True)

### 5.1 Price Trends - Top & Bottom Performers

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Normalized prices (rebased to 100)
normalized_prices = adj_close_filled / adj_close_filled.iloc[0] * 100

# Top 10
top_10 = metrics_sorted.head(10).index.tolist()
ax1 = axes[0]
for ticker in top_10:
    ax1.plot(normalized_prices.index, normalized_prices[ticker], label=ticker, linewidth=1.5)
ax1.set_title('Top 10 Performers - Normalized Price Trends', fontsize=14, fontweight='bold')
ax1.set_ylabel('Normalized Price (100 = Start)')
ax1.legend(loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.3)

# Bottom 10
bottom_10 = metrics_sorted.tail(10).index.tolist()
ax2 = axes[1]
for ticker in bottom_10:
    ax2.plot(normalized_prices.index, normalized_prices[ticker], label=ticker, linewidth=1.5)
ax2.set_title('Bottom 10 Performers - Normalized Price Trends', fontsize=14, fontweight='bold')
ax2.set_xlabel('Date')
ax2.set_ylabel('Normalized Price (100 = Start)')
ax2.legend(loc='upper left', fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{output_dir}/01_price_trends.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Risk-Return Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

scatter = ax.scatter(metrics['Volatility (%)'], 
                     metrics['Annualized Return (%)'],
                     s=metrics['Avg Volume (M)']*2,
                     c=metrics['Sharpe Ratio'],
                     cmap='RdYlGn',
                     alpha=0.7,
                     edgecolors='black',
                     linewidth=0.5)

# Add labels
for idx in metrics_sorted.head(10).index:
    ax.annotate(idx, 
                (metrics.loc[idx, 'Volatility (%)'], metrics.loc[idx, 'Annualized Return (%)']),
                fontsize=8, ha='center', va='bottom')

ax.axhline(y=0, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel('Volatility (Annualized %)', fontsize=12)
ax.set_ylabel('Annualized Return (%)', fontsize=12)
ax.set_title('Risk-Return Analysis (Size = Volume, Color = Sharpe Ratio)', fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Sharpe Ratio')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{output_dir}/03_risk_return.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Performance Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

sorted_metrics = metrics.sort_values('Total Return (%)', ascending=True)
colors = ['green' if x > 0 else 'red' for x in sorted_metrics['Total Return (%)']]

ax.barh(sorted_metrics.index, sorted_metrics['Total Return (%)'], color=colors, alpha=0.7, edgecolor='black')
ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
ax.set_xlabel('Total Return (%)', fontsize=12)
ax.set_ylabel('Stock Ticker', fontsize=12)
ax.set_title('Stock Performance - Total Return (%)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(f'{output_dir}/05_performance_bar.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.4 Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))

corr_matrix = daily_returns.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, 
            mask=mask,
            annot=False,
            cmap='coolwarm',
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8},
            ax=ax)
ax.set_title('Stock Returns Correlation Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{output_dir}/04_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.5 Volatility Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax1 = axes[0]
vol_data = metrics['Volatility (%)'].dropna()
ax1.hist(vol_data, bins=20, alpha=0.7, color='coral', edgecolor='black')
ax1.axvline(x=vol_data.mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {vol_data.mean():.1f}%")
ax1.set_xlabel('Volatility (Annualized %)')
ax1.set_ylabel('Number of Stocks')
ax1.set_title('Volatility Distribution')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
vol_sorted = metrics.sort_values('Volatility (%)', ascending=False).head(15)
ax2.barh(vol_sorted.index, vol_sorted['Volatility (%)'], color='coral', alpha=0.7, edgecolor='black')
ax2.set_xlabel('Volatility (Annualized %)')
ax2.set_title('Top 15 Most Volatile Stocks')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(f'{output_dir}/06_volatility_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary Statistics

In [ ]:
print("=" * 60)
print("MARKET OVERVIEW")
print("=" * 60)
print(f"\nAverage Return: {metrics['Annualized Return (%)'].mean():.2f}%")
print(f"Median Return: {metrics['Annualized Return (%)'].median():.2f}%")
print(f"Average Volatility: {metrics['Volatility (%)'].mean():.2f}%")
print(f"Average Sharpe Ratio: {metrics['Sharpe Ratio'].mean():.2f}")
print(f"Average Max Drawdown: {metrics['Max Drawdown (%)'].mean():.2f}%")

print("\n" + "=" * 60)
print("TOP 5 PERFORMERS")
print("=" * 60)
for i, (ticker, row) in enumerate(metrics_sorted.head(5).iterrows(), 1):
    print(f"{i}. {ticker}: +{row['Total Return (%)']:.2f}% (Sharpe: {row['Sharpe Ratio']:.2f})")

print("\n" + "=" * 60)
print("BOTTOM 5 PERFORMERS")
print("=" * 60)
for i, (ticker, row) in enumerate(metrics_sorted.tail(5).iloc[::-1].iterrows(), 1):
    print(f"{i}. {ticker}: {row['Total Return (%)']:.2f}% (Sharpe: {row['Sharpe Ratio']:.2f})")

## 7. Export Results

In [ ]:
# Save metrics to CSV
metrics_sorted.to_csv('sp500_metrics.csv')
print("Metrics saved to: sp500_metrics.csv")

# Save correlation matrix
corr_matrix.to_csv('sp500_correlation.csv')
print("Correlation matrix saved to: sp500_correlation.csv")

print("\nAll visualizations saved to:", output_dir)

---
## Conclusion

This EDA project demonstrates:
- **Data Acquisition:** Fetching real market data from Yahoo Finance
- **Data Cleaning:** Handling missing values and preprocessing
- **Statistical Analysis:** Calculating returns, volatility, Sharpe ratio, and max drawdown
- **Visualization:** Creating 8 professional charts for comprehensive analysis
- **Insights:** Identifying top performers, risk metrics, and market correlations

---
*Analysis completed using Python, pandas, matplotlib, and seaborn*